In [1]:
# =========================================
# 0. CREATE OUTPUT FOLDER
# =========================================
import os

output_folder = "finalComparison"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [2]:
# =========================================
# 1. IMPORT LIBRARIES
# =========================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from textblob import TextBlob
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import warnings
warnings.filterwarnings("ignore")

In [3]:
# =========================================
# 2. LOAD DATA
# =========================================
comments = pd.read_csv("Data/English_version/English_translated_comment_data.csv")
live = pd.read_csv("Data/English_version/English_translated_live_chat_data.csv")
videos = pd.read_csv("Data/English_version/English_translated_video_informationt.csv")

In [4]:
# =========================================
# 3. ADD CHANNEL TO COMMENTS (IMPORTANT)
# =========================================
# Merge with video data using video_id
comments = comments.merge(videos[['video_id','channel']], on='video_id', how='left')

In [5]:
# =========================================
# 4. TEXT CLEANING FUNCTION
# =========================================
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()



In [6]:
# =========================================
# 5. PREPARE TEXT DATA (COMMENTS + LIVE)
# =========================================
comments['clean'] = comments['translated_text'].apply(clean_text)
live['clean'] = live['translated_text'].apply(clean_text)

comments = comments[comments['clean'] != ""]
live = live[live['clean'] != ""]


In [7]:
# =========================================
# 6. SENTIMENT FUNCTION
# =========================================
def get_sentiment(text):
    blob = TextBlob(text)
    polarity = blob.sentiment.polarity

    if polarity > 0.1:
        return "Positive"
    elif polarity < -0.1:
        return "Negative"
    else:
        return "Neutral"


# Apply sentiment
comments['sentiment'] = comments['clean'].apply(get_sentiment)
live['sentiment'] = live['clean'].apply(get_sentiment)



In [8]:
# =========================================
# 7. SENTIMENT COMPARISON (ALL)
# =========================================
comments_sent = comments.groupby(['channel','sentiment']).size().unstack().fillna(0)
live_sent = live.groupby(['channel','sentiment']).size().unstack().fillna(0)

combined_sent = comments_sent.add(live_sent, fill_value=0)

print("\n===== COMBINED SENTIMENT =====")
print(combined_sent)


===== COMBINED SENTIMENT =====
sentiment  Negative  Neutral  Positive
channel                               
Ch-1          21512   266925     62540
Ch-2           6639   159174     27828


In [9]:
# =========================================
# 8. SENTIMENT GRAPH
# =========================================
combined_sent.plot(kind='bar', figsize=(8,5))
plt.title("Overall Sentiment Comparison (Ch-1 vs Ch-2)")
plt.tight_layout()
plt.savefig(f"{output_folder}/combined_sentiment.png")
plt.close()



In [10]:
# =========================================
# 9. TOPIC MODELING (COMBINED TEXT)
# =========================================
all_text = pd.concat([comments[['clean','channel']], live[['clean','channel']]])

vectorizer = CountVectorizer(stop_words='english', min_df=2)
dtm = vectorizer.fit_transform(all_text['clean'])

lda = LatentDirichletAllocation(n_components=3, random_state=42)
lda.fit(dtm)

doc_topics = lda.transform(dtm)
all_text['Topic'] = np.argmax(doc_topics, axis=1)

topic_labels = {
    0: "Positive Feedback",
    1: "Requests/Suggestions",
    2: "Emotional Response"
}
all_text['Topic_Label'] = all_text['Topic'].map(topic_labels)

In [11]:
# =========================================
# 10. TOPIC COMPARISON
# =========================================
topic_summary = all_text.groupby(['channel','Topic_Label']).size().unstack().fillna(0)

print("\n===== COMBINED TOPICS =====")
print(topic_summary)




===== COMBINED TOPICS =====
Topic_Label  Emotional Response  Positive Feedback  Requests/Suggestions
channel                                                                 
Ch-1                     134245             100623                116109
Ch-2                      50080              95482                 48079


In [12]:
# =========================================
# 11. TOPIC GRAPH
# =========================================
topic_summary.plot(kind='bar', figsize=(8,5))
plt.title("Overall Topic Comparison")
plt.tight_layout()
plt.savefig(f"{output_folder}/combined_topics.png")
plt.close()



In [13]:
# =========================================
# 12. VIDEO PERFORMANCE
# =========================================
videos['engagement'] = (videos['video_likes'] + videos['comment_count']) / videos['view_count']

video_summary = videos.groupby('channel')[['view_count','comment_count','video_likes','engagement']].mean()

print("\n===== VIDEO PERFORMANCE =====")
print(video_summary)



===== VIDEO PERFORMANCE =====
            view_count  comment_count   video_likes  engagement
channel                                                        
Ch-1     696017.434524    1028.952381  16431.660714    0.026473
Ch-2     863007.306452    1490.483871  23060.822581    0.029924


In [14]:
# =========================================
# 13. VIDEO GRAPH
# =========================================
video_summary[['view_count','video_likes']].plot(kind='bar')
plt.title("Video Performance Comparison")
plt.tight_layout()
plt.savefig(f"{output_folder}/video_performance.png")
plt.close()


In [15]:
# =========================================
# 14. SAVE ALL OUTPUTS
# =========================================
combined_sent.to_csv(f"{output_folder}/combined_sentiment.csv")
topic_summary.to_csv(f"{output_folder}/combined_topics.csv")
video_summary.to_csv(f"{output_folder}/video_summary.csv")

print("\n✅ FINAL COMBINED ANALYSIS COMPLETE!")


✅ FINAL COMBINED ANALYSIS COMPLETE!
